# Identifying Splice-Variants and Post-Prenylation Processing Variants in MaxQuant Results
The Protein Groups output tables from MaxQuant will be filtered by the fasta identifiers for isoforms (N_term, Internal, C_term) and prenylation variants (INTERNAL_PAE0, INTERNAL_LOWPAE und INTERNAL_C1).

# Setup

## Import and install Packages

In [ ]:
import sys
import os
import session_info

# Add the '0_functions' folder to sys.path
sys.path.append(os.path.join(os.getcwd(), '..', '00_functions'))

In [ ]:
import pandas as pd
import re
import openpyxl
from functions import read_fastafile

In [ ]:
# Display session information
session_info.show()

## Set working directory

In [ ]:
cwd = os.getcwd()
if not cwd.endswith("Isoform_Database/Jupyter_environment/Isoform_Database_SIAF/05_MaxQuant"):
    print("WARNING: The working directory is not set to the '05_MaxQuant'!")
    print("Current working directory:", cwd)
else:
    print(f"Working directory is correctly set to the '05_MaxQuant' folder (\"{cwd}\").")

In [ ]:
# Data directories
prenylation_dir = "../04_Prenylation_Database/data/"
mapping_dir = "../01_UniProt/data/mapping/"
isoform_database_dir = "../02_Isoform_Database/"

total_extracts_dir = "data/total_extracts/"
prenylation_enrichment_dir = "data/prenylation_enrichment/"
peptides_dir = "data/peptides/"

## Read in MaxQuant Results and Lists of isoforms and prenylated proteins

In [ ]:
isoform_fasta = read_fastafile(os.path.join(isoform_database_dir, 'Isoform_Database_only_iso.fasta'))
prenylation_fasta = read_fastafile(os.path.join(prenylation_dir, 'Post-Prenylation_Processing_Database.fasta'))
canonical_fasta = read_fastafile(os.path.join(isoform_database_dir, 'unique_canonical.fasta'))
motif_2_list = pd.read_csv(os.path.join(prenylation_dir, 'motif_2_list.csv'))
iso_canonical_mapping = pd.read_csv(os.path.join(mapping_dir, 'iso_canonical_mapping.csv')) 

#total extracts
mq_results_act_te = pd.read_excel(os.path.join(total_extracts_dir, 'proteinGroups_act_te_nrdbln.xlsx')) # Th1-Cells activated vs non-activated
mq_results_stat_te = pd.read_excel(os.path.join(total_extracts_dir, 'proteinGroups_stat_te_nrdbln.xlsx')) # activated Th1 cells +/- statin treatment
mq_results_FTI_te = pd.read_excel(os.path.join(total_extracts_dir, 'proteinGroups_FTI_te_nrdbln.xlsx')) # activated Th1-Cells +/- treatment with farnesyl transferase inhibitor

#prenylation enriched (with clickit-reaction)
mq_results_act_clickit = pd.read_excel(os.path.join(prenylation_enrichment_dir, 'protein_Groups_act_clickit_nrdbln.xlsx')) # Th1-Cells activated vs non-activated
mq_results_stat_clickit = pd.read_excel(os.path.join(prenylation_enrichment_dir, 'protein_Groups_stat_clickit_nrdbln.xlsx')) # activated Th1 cells +/- statin treatment
mq_results_FTI_clickit = pd.read_excel(os.path.join(prenylation_enrichment_dir, 'protein_Groups_FTI_clickit_nrdbln.xlsx')) # activated Th1-Cells +/- treatment with farnesyl transferase inhibitor

# Filter Fasta headers by Isoform and Prenylation fasta headers

In [ ]:
isoform_set = set(isoform_fasta['seqID'])
prenylation_set = set(prenylation_fasta['seqID'])
motif_2_set = set(canonical_fasta.loc[canonical_fasta['ID'].isin(motif_2_list['Protein_ID']), 'seqID'])

In [ ]:
def filter_maxquant_variants(df, isoform_set, prenylation_set, motif_2_set,
                             id_col='Protein IDs', 
                             header_col='Fasta headers', 
                             min_unique_peptides=1):
    """
    Cleans MaxQuant results and splits them into Isoforms and Prenylation variants
    using statistical thresholds for FDR and peptide count.
    """
    # Basic Cleaning (Decoys, Contaminants, and Only identified by site)
    initial_count = len(df)
    
    # Remove rows based on MaxQuant flag columns
    flags = ['Reverse', 'Potential contaminant', 'Only identified by site']
    for col in flags:
        if col in df.columns:
            df = df[df[col] != '+']
    
    # Filter by Unique Peptides
    if 'Unique peptides' in df.columns:
        df = df[df['Unique peptides'] >= min_unique_peptides]

    clean_df = df.copy()
    
    # 3. Helper function for set intersection
    def check_match(header_str, target_set):
        if pd.isna(header_str): return False
        # Split by semicolon as MaxQuant often concatenates IDs
        return any(h.strip() in target_set for h in str(header_str).split(';'))

    # 4. Create the two filtered DataFrames
    df_isoforms = clean_df[
        clean_df[header_col].apply(lambda x: check_match(x, isoform_set))
    ].copy()
    
    df_prenylation_processing = clean_df[
        clean_df[header_col].apply(lambda x: check_match(x, prenylation_set))
    ].copy()

    df_prenylation_cterm = clean_df[
        clean_df[header_col].apply(lambda x: check_match(x, motif_2_set))
    ].copy()

    # Reporting
    print(f"--- Filter Report ---")
    print(f"Initial Rows: {initial_count}")
    print(f"Isoform hits: {len(df_isoforms)}")
    print(f"Post Prenylation Processing hits: {len(df_prenylation_processing)}")
    print(f"Internal motif hits: {len(df_prenylation_cterm)}")
    
    return df_isoforms, df_prenylation_processing, df_prenylation_cterm

In [ ]:
# Execution:
#total extracts
isoforms_act_te, prenylation_act_te, pren_cterm_act_te = filter_maxquant_variants(mq_results_act_te, isoform_set, prenylation_set, motif_2_set)
isoforms_stat_te, prenylation_stat_te, pren_cterm_stat_te = filter_maxquant_variants(mq_results_stat_te, isoform_set, prenylation_set, motif_2_set)
isoforms_FTI_te, prenylation_FTI_te, pren_cterm_FTI_te = filter_maxquant_variants(mq_results_FTI_te, isoform_set, prenylation_set, motif_2_set)

#prentylation enrichment
isoforms_act_clickit, prenylation_act_clickit, pren_cterm_act_clickit = filter_maxquant_variants(mq_results_act_clickit, isoform_set, prenylation_set, motif_2_set)
isoforms_stat_clickit, prenylation_stat_clickit, pren_cterm_stat_clickit = filter_maxquant_variants(mq_results_stat_clickit, isoform_set, prenylation_set, motif_2_set)
isoforms_FTI_clickit, prenylation_FTI_clickit, pren_cterm_FTI_clickit = filter_maxquant_variants(mq_results_FTI_clickit, isoform_set, prenylation_set, motif_2_set)

In [ ]:
# Save splice variants as csv 
isoforms_act_te.to_csv(os.path.join(total_extracts_dir, 'MQ_splice_variants_act_te.csv'), index=False)
isoforms_stat_te.to_csv(os.path.join(total_extracts_dir, 'MQ_splice_variants_stat_te.csv'), index=False)
isoforms_FTI_te.to_csv(os.path.join(total_extracts_dir, 'MQ_splice_variants_FTI_te.csv'), index=False)

isoforms_act_clickit.to_csv(os.path.join(prenylation_enrichment_dir, 'MQ_splice_variants_act_clickit.csv'), index=False)
isoforms_stat_clickit.to_csv(os.path.join(prenylation_enrichment_dir, 'MQ_splice_variants_stat_clickit.csv'), index=False)
isoforms_FTI_clickit.to_csv(os.path.join(prenylation_enrichment_dir, 'MQ_splice_variants_FTI_clickit.csv'), index=False)

In [ ]:
print(len(isoforms_act_te))
print(len(isoforms_stat_te))
print(len(isoforms_FTI_te))

print(len(isoforms_act_clickit))
print(len(isoforms_stat_clickit))
print(len(isoforms_FTI_clickit))

In [ ]:
# Save post-prenylation processing variants as csv 
prenylation_act_te.to_csv(os.path.join(total_extracts_dir, 'MQ_prenylation_variants_act_te.csv'), index=False)
prenylation_stat_te.to_csv(os.path.join(total_extracts_dir, 'MQ_prenylation_variants_stat_te.csv'), index=False)
prenylation_FTI_te.to_csv(os.path.join(total_extracts_dir, 'MQ_prenylation_variants_FTI_te.csv'), index=False)

prenylation_act_clickit.to_csv(os.path.join(prenylation_enrichment_dir, 'MQ_prenylation_variants_act_clickit.csv'), index=False)    
prenylation_stat_clickit.to_csv(os.path.join(prenylation_enrichment_dir, 'MQ_prenylation_variants_stat_clickit.csv'), index=False)
prenylation_FTI_clickit.to_csv(os.path.join(prenylation_enrichment_dir, 'MQ_prenylation_variants_FTI_clickit.csv'), index=False)    

In [ ]:
# Save internal motif proteins as csv
pren_cterm_act_te.to_csv(os.path.join(total_extracts_dir, 'MQ_pren_cterm_act_te.csv'), index=False)
pren_cterm_stat_te.to_csv(os.path.join(total_extracts_dir, 'MQ_pren_cterm_stat_te.csv'), index=False)
pren_cterm_FTI_te.to_csv(os.path.join(total_extracts_dir, 'MQ_pren_cterm_FTI_te.csv'), index=False)

pren_cterm_act_clickit.to_csv(os.path.join(prenylation_enrichment_dir, 'MQ_pren_cterm_act_clickit.csv'), index=False)  
pren_cterm_stat_clickit.to_csv(os.path.join(prenylation_enrichment_dir, 'MQ_pren_cterm_stat_clickit.csv'), index=False)  
pren_cterm_FTI_clickit.to_csv(os.path.join(prenylation_enrichment_dir, 'MQ_pren_cterm_FTI_clickit.csv'), index=False)  

# Internal Prenylation motifs
We will check if for the proteins with an internal prenylation motif, any tryptic peptides more C-terminal than the potentially prenylated cysteine are found.

In [ ]:
df_prenylation = pd.read_excel(os.path.join(prenylation_dir,'SupplementaryTableS1.xlsx'))

# peptides te
peptides_act_te = pd.read_excel(os.path.join(peptides_dir, 'peptides_act_te_nrdbln.xlsx'))
peptides_stat_te = pd.read_excel(os.path.join(peptides_dir, 'peptides_stat_te_nrdbln.xlsx'))
peptides_FTI_te = pd.read_excel(os.path.join(peptides_dir, 'peptides_FTI_te_nrdbln.xlsx'))

#peptides clickit
peptides_act_clickit = pd.read_excel(os.path.join(peptides_dir, 'peptides_act_clickit_nrdbln.xlsx'))
peptides_stat_clickit = pd.read_excel(os.path.join(peptides_dir, 'peptides_stat_clickit_nrdbln.xlsx'))
peptides_FTI_clickit = pd.read_excel(os.path.join(peptides_dir, 'peptides_FTI_clickit_nrdbln.xlsx'))

In [ ]:
motif_2 = df_prenylation[df_prenylation['Motif'] == 2].copy()

In [ ]:
# Add peptide length to motif_2 df
length_map = canonical_fasta.set_index('ID')['len']
motif_2['peptide_length'] = motif_2['Protein_ID'].map(length_map)

In [ ]:
# remove columns not needed to make df cleaner
motif_2 = motif_2.drop(columns=motif_2.columns[2:23])

In [ ]:
# create N-term pos and C-term pos columns

cols_to_combine = ['PositionC_Internal_C1',
       'PositionC_Internal_C_wo_disulfidebond', 'PositionC_Internal_pae0',
       'PositionC_Internal_Lowest_pae']
motif_2['N-term_pos'] = motif_2[cols_to_combine].bfill(axis=1).iloc[:, 0]

motif_2['C-term_pos'] = motif_2['peptide_length'] + motif_2['N-term_pos'] + 1

In [ ]:
motif_2[motif_2['Protein_ID'] == 'P09110']

In [ ]:
# Extract protein id from the middle part between the '|' characters
# te
peptides_act_te['protein_id'] = peptides_act_te['Proteins'].str.split('|').str[1]
peptides_stat_te['protein_id'] = peptides_stat_te['Proteins'].str.split('|').str[1]
peptides_FTI_te['protein_id'] = peptides_FTI_te['Proteins'].str.split('|').str[1]

# clickit
peptides_act_clickit['protein_id'] = peptides_act_clickit['Proteins'].str.split('|').str[1]
peptides_stat_clickit['protein_id'] = peptides_stat_clickit['Proteins'].str.split('|').str[1]
peptides_FTI_clickit['protein_id'] = peptides_FTI_clickit['Proteins'].str.split('|').str[1]

In [ ]:
def get_proteins_after_cysteine(
    peptide_df, 
    target_df, 
    peptide_id_col='protein_id', 
    peptide_start_col='Start position', 
    target_id_col='Protein_ID', 
    c_pos_col='N-term_pos'
):

    # 1. Merge the dataframes on the protein ID columns
    merged_df = target_df.merge(
        peptide_df[[peptide_id_col, peptide_start_col]], 
        left_on=target_id_col, 
        right_on=peptide_id_col, 
        how='inner'
    )
    
    # 2. Filter rows where the peptide start position is greater than (after) the cysteine position
    matching_rows = merged_df[merged_df[peptide_start_col] > merged_df[c_pos_col]]
    
    # 3. Extract the unique protein IDs into a list
    protein_list = matching_rows[target_id_col].unique().tolist()
    
    return protein_list

In [ ]:
# Call the function
# te
after_C_act_te = get_proteins_after_cysteine(
    peptide_df=peptides_act_te,
    target_df=motif_2,
    peptide_id_col='protein_id',       
    peptide_start_col='Start position',          
    target_id_col='Protein_ID',        
    c_pos_col='N-term_pos'            
)
after_C_stat_te = get_proteins_after_cysteine(
    peptide_df=peptides_stat_te,
    target_df=motif_2,
    peptide_id_col='protein_id',       
    peptide_start_col='Start position',          
    target_id_col='Protein_ID',        
    c_pos_col='N-term_pos'            
)
after_C_FTI_te = get_proteins_after_cysteine(
    peptide_df=peptides_FTI_te,
    target_df=motif_2,
    peptide_id_col='protein_id',       
    peptide_start_col='Start position',          
    target_id_col='Protein_ID',        
    c_pos_col='N-term_pos'            
)

# clickit
after_C_act_clickit = get_proteins_after_cysteine(
    peptide_df=peptides_act_clickit,
    target_df=motif_2,
    peptide_id_col='protein_id',       
    peptide_start_col='Start position',          
    target_id_col='Protein_ID',        
    c_pos_col='N-term_pos'            
)
after_C_stat_clickit = get_proteins_after_cysteine(
    peptide_df=peptides_stat_clickit,
    target_df=motif_2,
    peptide_id_col='protein_id',       
    peptide_start_col='Start position',          
    target_id_col='Protein_ID',        
    c_pos_col='N-term_pos'            
)
after_C_FTI_clickit = get_proteins_after_cysteine(
    peptide_df=peptides_FTI_clickit,
    target_df=motif_2,
    peptide_id_col='protein_id',       
    peptide_start_col='Start position',          
    target_id_col='Protein_ID',        
    c_pos_col='N-term_pos'            
)

In [ ]:
print(len(after_C_act_te))
print(len(after_C_stat_te))
print(len(after_C_FTI_te))

print(len(after_C_act_clickit))
print(len(after_C_stat_clickit))
print(len(after_C_FTI_clickit))